# Checking EC0 and EC50 ND4 doubling times

In [10]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
import numpy as np
import napari
import os

In [2]:
napari.__version__

'0.6.4'

In [3]:
napari.Viewer()

Viewer(camera=Camera(center=(0.0, 0.0, 0.0), zoom=1.0, angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(1.0, 1.0), viewbox=None, scaled=True, style=<CursorStyle.STANDARD: 'standard'>, size=1.0), dims=Dims(ndim=2, ndisplay=2, order=(0, 1), axis_labels=('0', '1'), rollable=(True, True), range=(RangeTuple(start=0.0, stop=2.0, step=1.0), RangeTuple(start=0.0, stop=2.0, step=1.0)), margin_left=(0.0, 0.0), margin_right=(0.0, 0.0), point=(np.float64(0.0), np.float64(0.0)), last_used=0), grid=GridCanvas(stride=1, shape=(-1, -1), enabled=False, spacing=0.0), layers=[], help='', status='Ready', tooltip=Tooltip(visible=False, text=''), theme='dark', title='napari', mouse_over_canvas=False, mouse_move_callbacks=[], mouse_drag_callbacks=[<function drag_to_zoom at 0x16e5097e0>], mouse_double_click_callbacks=[<f

## Checking growth model on ND4

In [146]:
df = pd.read_pickle('/Users/dayn/data/macrohet_mac/sc_df.pkl')

In [147]:
df

,Time (hours),Mtb Area (µm),dMtb Area (µm),Mphi Area (µm),dMphi Area (µm),Infection Status,Initial Infection Status,Final Infection Status,x,y,...,Edge Status,dMtb Area between frames (µm),Mtb Area Processed (µm),Time Model (hours),Mtb Area Model (µm),mtb_origin,Doubling Amounts,Doubling Times,r2,Frame
0,0.0,0.0,85.30408,914.186190,980.270593,0.0,0.0,1.0,716.326599,843.3125,...,NaN,NaN,0.0,0.0,0.0,Undefined,"[3.84, 7.68, 15.36, 30.72, 61.44]","[2.2, 0.0, 0.1, 0.2, 19.0]",0.999188,0
1,0.5,NaN,85.30408,NaN,980.270593,NaN,0.0,1.0,716.326599,843.3125,...,NaN,NaN,0.0,0.5,0.0,Undefined,"[3.84, 7.68, 15.36, 30.72, 61.44]","[2.2, 0.0, 0.1, 0.2, 19.0]",0.999188,1
2,1.0,NaN,85.30408,NaN,980.270593,NaN,0.0,1.0,716.326599,843.3125,...,NaN,NaN,0.0,1.0,0.0,Undefined,"[3.84, 7.68, 15.36, 30.72, 61.44]","[2.2, 0.0, 0.1, 0.2, 19.0]",0.999188,2
3,1.5,NaN,85.30408,NaN,980.270593,NaN,0.0,1.0,716.326599,843.3125,...,NaN,NaN,0.0,1.5,0.0,Undefined,"[3.84, 7.68, 15.36, 30.72, 61.44]","[2.2, 0.0, 0.1, 0.2, 19.0]",0.999188,3
4,2.0,NaN,85.30408,NaN,980.270593,NaN,0.0,1.0,716.326599,843.3125,...,NaN,NaN,0.0,2.0,0.0,Undefined,"[3.84, 7.68, 15.36, 30.72, 61.44]","[2.2, 0.0, 0.1, 0.2, 19.0]",0.999188,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2007856,59.5,0.0,NaN,261.387613,NaN,False,0.0,0.0,1034.000000,1161.0000,...,False,NaN,0.0,58.5,0.0,Undefined,[],[],-1.02,119
2007857,60.0,0.0,NaN,275.109568,NaN,False,0.0,0.0,972.000000,1147.0000,...,False,NaN,0.0,59.0,0.0,Undefined,[],[],-1.02,120
2007858,60.5,0.0,NaN,404.484815,NaN,False,0.0,0.0,948.000000,1159.0000,...,False,NaN,0.0,59.5,0.0,Undefined,[],[],-1.02,121
2007859,61.0,0.0,NaN,297.055758,NaN,NaN,0.0,0.0,874.000000,1163.0000,...,False,NaN,0.0,NaN,NaN,Undefined,[],[],-1.02,122


In [148]:
subset_df = df[df['mtb_origin']=='Growth'].drop_duplicates('ID').explode('Doubling Times').dropna(subset=['Doubling Times'])
subset_df = subset_df[subset_df['Doubling Times'] > 0]
subset_df = subset_df[(subset_df['Strain'] == 'WT') |  # only rd1 control if rd1
               ((subset_df['Strain'] == 'RD1') & (subset_df['Concentration'] == 'EC0'))]
print(len(subset_df))

760


In [149]:
subset_df['Doubling Times'].min()

np.float64(4.0)

# Doubling Calculations

In [118]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def calculate_dynamic_doubling(sub_df):
    """
    Calculates robust doubling times AND amounts for a single ID using 
    dynamic baseline logic (starts at actual size if >1.92).
    
    Returns:
        (list, list): (doubling_times, doubling_amounts)
    """
    # Sort to ensure time is strictly increasing
    sub_df = sub_df.sort_values(by='Time Model (hours)')
    
    # Extract arrays
    times = sub_df['Time Model (hours)'].values
    area_model = sub_df['Mtb Area Model (µm)'].values
    
    # Safety: need at least 2 points to calc anything
    if len(times) < 2:
        return [], []

    # 1. Determine Start Baseline
    start_area = area_model[0] if not np.isnan(area_model[0]) else 1.92
    baseline = max(1.92, start_area)
    
    # 2. Generate Grid (Baseline -> 2x -> 4x -> 8x...)
    grid = [baseline * (2**i) for i in range(1, 6)] # Start from 2x
    
    # 3. Crossing Helper (Strict & Robust)
    def find_crossing(target, t_arr, a_arr):
        valid_mask = ~np.isnan(a_arr)
        clean_a = a_arr[valid_mask]
        
        # Strict existence check
        if not np.any(clean_a >= target): 
            return None
        
        # Fill NaNs for argmax safety
        filled_a = np.nan_to_num(a_arr, nan=-np.inf)
        idx = np.argmax(filled_a >= target)
        
        # Safety for index 0 (prevent 'Ghost' crossing)
        if idx == 0 and filled_a[0] < target:
            return None
        if idx == 0: return t_arr[0]
        
        # Linear Interpolation
        t1, t2 = t_arr[idx-1], t_arr[idx]
        a1, a2 = filled_a[idx-1], filled_a[idx]
        if a2 == a1: return t1
        fraction = (target - a1) / (a2 - a1)
        return t1 + (t2 - t1) * fraction

    # 4. Calculate Intervals & Amounts
    calc_intervals = []
    calc_amounts = []
    prev_time = times[0]
    
    # Determine effective start time for the first interval
    # (If we started < 1.92, we need the time we crossed 1.92, not t[0])
    if start_area < baseline:
        t_start_real = find_crossing(baseline, times, area_model)
        if t_start_real is not None:
            prev_time = t_start_real
        else:
            return [], [] # Never even reached baseline
    
    for target in grid:
        t_cross = find_crossing(target, times, area_model)
        
        if t_cross is not None:
            # Noise filter: time must move forward
            if t_cross >= prev_time:
                dt = t_cross - prev_time
                
                # Record Data
                calc_intervals.append(round(dt, 1))
                calc_amounts.append(round(target, 2)) # Record the target amount reached
                
                prev_time = t_cross
            else:
                break # Stop if we hit noise (time went backward)
        else:
            break # Stop if target not reached
            
    return calc_intervals, calc_amounts

# --- APPLY TO DATAFRAME ---

# 1. Initialize columns as object type
df['Doubling Timez'] = np.nan
df['Doubling Timez'] = df['Doubling Timez'].astype(object)

df['Doubling Amountz'] = np.nan
df['Doubling Amountz'] = df['Doubling Amountz'].astype(object)

# 2. Iterate by ID
ids = df['ID'].unique()
total = len(ids)

print(f"Processing {total} IDs...")

for i, uid in tqdm(enumerate(ids), total=len(ids)):
    mask = df['ID'] == uid
    sub_df = df[mask]
    
    # Calculate both lists
    dts, amts = calculate_dynamic_doubling(sub_df)
    
    # Assign results carefully
    # We use a loop or specific index assignment to handle the list-in-cell assignment
    idx = df[mask].index
    
    # Create an array of objects to assign at once (faster than row-by-row)
    # We need a list of lists: [[dt, dt], [dt, dt]...] matching the mask length
    dt_series = np.empty(len(idx), dtype=object)
    dt_series[:] = [dts] * len(idx)
    
    amt_series = np.empty(len(idx), dtype=object)
    amt_series[:] = [amts] * len(idx)
    
    df.loc[idx, 'Doubling Timez'] = dt_series
    df.loc[idx, 'Doubling Amountz'] = amt_series

print("Done! 'Doubling Timez' and 'Doubling Amountz' columns created.")

Processing 19688 IDs...


  0%|          | 0/19688 [00:00<?, ?it/s]

Done! 'Doubling Timez' and 'Doubling Amountz' columns created.


# have a look at new dts calcs

In [139]:
IDs = subset_df.sort_values('Doubling Times').ID.values

In [140]:
annotator = GlimpseAnnotator(df, IDs)

--- Processing 1/779: 1003.4.10.ND0003 ---
--- Processing 2/779: 1235.4.9.ND0004 ---
--- Processing 3/779: 164.3.5.ND0002 ---
--- Processing 4/779: 2073.4.7.ND0004 ---
--- Processing 5/779: 1003.4.10.ND0003 ---
--- Processing 6/779: 8022.4.9.ND0004 ---
--- Processing 7/779: 601.4.5.PS0000 ---
--- Processing 8/779: 1235.4.9.ND0004 ---


# Class

In [134]:
import napari
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import imageio.v3 as iio
import os
import io
import imageio  # This gives access to the v2 API
from matplotlib.lines import Line2D
import re   

# --- Configuration ---
VIDEO_DIR = '/Volumes/OPERA3/Nathan/data/macrohet/macrohet_results/glimpse_store/glimpses/'
SAVE_PATH = '/Volumes/OPERA3/Nathan/data/macrohet/macrohet_results/sc_df_amount_tests.pkl'

# --- Ensure Style Defaults ---
sns.set_style("white")

class GlimpseAnnotator:
    def __init__(self, full_df, id_list):
        self.df = full_df         
        self.id_list = id_list    
        self.current_idx = 0
        self.total = len(id_list)
        
        # Initialize Viewer
        self.viewer = napari.Viewer(title="Glimpse Annotator")
        
        # Setup bindings and load first sample
        self.setup_bindings()
        self.load_sample()

    def load_sample(self):
        """
        Loads video and plot. 
        AUTOMATICALLY SKIPS to the next ID if the video file is missing.
        """
        # Safety Check: Stop if we reached the end
        if self.current_idx >= self.total:
            print("All samples processed!")
            return

        self.current_id = self.id_list[self.current_idx]
        video_path = os.path.join(VIDEO_DIR, f'{self.current_id}.mp4')

        # --- SKIPPING LOGIC ---
        if not os.path.exists(video_path):
            print(f"⚠️ Missing Video for {self.current_id}. Skipping...")
            self.current_idx += 1
            # Recursively call this function again to try the next ID
            self.load_sample() 
            return
        # ----------------------
        
        # Clear existing layers
        self.viewer.layers.select_all()
        self.viewer.layers.remove_selected()
        
        print(f"--- Processing {self.current_idx + 1}/{self.total}: {self.current_id} ---")

        # 1. Attempt to Load Video
        video_loaded = False
        try:
            reader = imageio.get_reader(video_path, format='ffmpeg')
            video = np.stack([frame for frame in reader])
            reader.close()
            
            self.viewer.add_image(video, name=f'{self.current_id} Video')
            video_loaded = True
        except Exception as e:
            print(f"Warning: Could not read video for {self.current_id}. Error: {e}")

        # 2. Generate and Add Plot
        try:
            plot_img = self.create_plot_image(self.current_id)
            if plot_img is not None:
                self.viewer.add_image(plot_img, name='Growth Plot')
            else:
                print(f"No plot data generated for {self.current_id}")
        except Exception as e:
            print(f"Error generating plot for {self.current_id}: {e}")

        # 3. Layout and Time Reset
        self.viewer.grid.enabled = True
        self.viewer.grid.shape = (1, 2) if video_loaded else (1, 1)
        self.viewer.reset_view()
        
        # Reset Time Slider to 0 (t=0)
        if self.viewer.dims.ndim > 0:
            current_step = list(self.viewer.dims.current_step)
            current_step[0] = 0
            self.viewer.dims.current_step = tuple(current_step)

    def create_plot_image(self, ID):
        """
        Generates the Matplotlib graph using the renewed 'Shifted Labels' approach.
        Plots shifted labels on the 'floor' of the interval.
        """
        # 1. Filter and Sort
        sub_df = self.df[self.df['ID'] == ID].copy()
        if sub_df.empty: return None
        sub_df = sub_df.sort_values(by='Time Model (hours)')

        # Create Figure
        fig, ax = plt.subplots(figsize=(7, 5), dpi=150)
        
        try:
            # --- Extract Data ---
            times = sub_df['Time Model (hours)'].values
            area_model = sub_df['Mtb Area Model (µm)'].values
            area_proc = sub_df['Mtb Area Processed (µm)'].values
            
            meta = sub_df.iloc[0]
            strain = meta.get('Strain', 'Unknown')
            compound = meta.get('Compound', 'N/A')
            conc = meta.get('Concentration', '')
            
            # --- Live Calculation ---
            # 1. Determine Start Baseline
            raw_start_area = area_model[0] if not np.isnan(area_model[0]) else 1.92
            baseline = max(1.92, raw_start_area)
            
            # 2. Helper
            def find_crossing(target, t_arr, a_arr):
                if not np.any(a_arr >= target): return None
                filled_a = np.nan_to_num(a_arr, nan=-np.inf)
                idx = np.argmax(filled_a >= target)
                if idx == 0 and filled_a[0] < target: return None
                if idx == 0: return t_arr[0]
                t1, t2 = t_arr[idx-1], t_arr[idx]
                a1, a2 = filled_a[idx-1], filled_a[idx]
                if a2 == a1: return t1
                return t1 + (t2 - t1) * ((target - a1) / (a2 - a1))

            # 3. Find Crossings
            # Handle start time for baseline
            if raw_start_area < baseline:
                t_start = find_crossing(baseline, times, area_model)
            else:
                t_start = times[0]
                
            if t_start is None:
                crossings_x = []
                crossings_y = []
            else:
                crossings_x = [t_start]
                crossings_y = [baseline]

            grid = [baseline * (2**i) for i in range(1, 6)]
            
            for target in grid:
                t_cross = find_crossing(target, times, area_model)
                if t_cross is not None:
                    if t_cross >= crossings_x[-1]:
                        crossings_x.append(t_cross)
                        crossings_y.append(target)
                    else:
                        break
                else:
                    break

            # --- Plotting ---
            ax.plot(times, area_model, color='#d02c91', lw=2.5, label='Model')
            ax.scatter(times, area_proc, color='#1a9641', s=20, alpha=0.6, label='Data')
            
            # Draw the initial Baseline (Floor of the first interval)
            if crossings_x:
                ax.axhline(y=baseline, color='lightgrey', linestyle='--', lw=1.5)
                ax.axvline(x=crossings_x[0], color='lightgrey', linestyle='--', lw=1.5)
            
            # --- Loop through intervals ---
            for i in range(1, len(crossings_x)):
                t_prev, t_curr = crossings_x[i-1], crossings_x[i]
                y_prev, y_curr = crossings_y[i-1], crossings_y[i]
                
                # Draw the "Ceiling" Grid Lines for this interval
                ax.axhline(y=y_curr, color='lightgrey', linestyle='--', lw=1.5)
                ax.axvline(x=t_curr, color='lightgrey', linestyle='--', lw=1.5)
                
                # --- CALCULATE VALUES ---
                dt = t_curr - t_prev
                delta_y = y_curr - y_prev
                
                # --- LABEL PLACEMENT ---
                # Placed on the "Floor" (y_prev) with a slight offset so it sits ON the line
                label_text = f"∆T = {dt:.1f}h | ∆Mtb = {delta_y:.2f} µm²"
                
                ax.text(t_prev + 0.2, 
                        y_prev * 1.02,  # Uses y_prev (Floor)
                        label_text, 
                        color='#505050', fontsize=9, fontweight='bold', ha='left', va='bottom',
                        bbox=dict(boxstyle='square,pad=0.2', fc='white', ec='none', alpha=0.7))

            # --- Formatting ---
            title = (f"ID: {ID}\n"
                     f"{strain} | {compound} {conc}")
            ax.set_title(title, fontsize=12)
            ax.set_xlabel("Time (hours)")
            ax.set_ylabel("Mtb Area (µm²)")
            
            custom_lines = [
                Line2D([0], [0], color='#d02c91', lw=2),
                Line2D([0], [0], color='lightgrey', linestyle='--', lw=1.5),
            ]
            ax.legend(custom_lines, ['Model', 'Doubling Grid'], loc='upper left')
            
            sns.despine()
            
            # --- Convert to Image for Napari ---
            buf = io.BytesIO()
            fig.savefig(buf, format="png", bbox_inches='tight')
            plt.close(fig)
            buf.seek(0)
            img = imageio.v3.imread(buf, index=0)
            return img
            
        except Exception as e:
            print(f"Plotting error for {ID}: {e}")
            plt.close(fig)
            return None

    def next_sample(self, viewer):
        self.current_idx += 1
        self.load_sample()

    def prev_sample(self, viewer):
        if self.current_idx > 0:
            self.current_idx -= 1
            self.load_sample()

    # --- Classification Actions ---
    def update_origin(self, status):
        # Update the main DF
        self.df.loc[self.df['ID'] == self.current_id, 'mtb_origin'] = status
        print(f"Set {self.current_id} -> {status}")

    def mark_transfer(self, viewer): self.update_origin('Transfer')
    def mark_uptake(self, viewer): self.update_origin('Uptake')
    def mark_junk(self, viewer): self.update_origin('Junk')
    def mark_growth(self, viewer): self.update_origin('Growth')
    
    def mark_edge(self, viewer):
        self.df.loc[self.df['ID'] == self.current_id, 'Edge Status'] = True
        print(f"ID {self.current_id} marked as 'Edge Status'.")

    def save_df(self, viewer):
        print("Saving dataframe...")
        tmp_path = SAVE_PATH + '.tmp'
        self.df.to_pickle(tmp_path)
        os.replace(tmp_path, SAVE_PATH)
        print(f"Saved to {SAVE_PATH}")

    def split_track(self, viewer):
        """
        Splits the track with HEAVY debugging output to diagnose failures.
        """
        print("\n" + "="*40)
        print(f"DEBUG: Attempting Split on {self.current_id}")
        
        # 1. Get relative frame from slider
        try:
            current_frame = viewer.dims.current_step[0]
            print(f"DEBUG: Slider Frame: {current_frame}")
        except Exception as e:
            print(f"ERROR: Could not get frame from viewer: {e}")
            return
        
        # 2. Find Track Start
        track_df = self.df[self.df['ID'] == self.current_id]
        if track_df.empty:
            print(f"ERROR: ID {self.current_id} not found in DataFrame!")
            return
            
        start_time = track_df['Time (hours)'].min()
        end_time = track_df['Time (hours)'].max()
        split_time = start_time + (current_frame * 0.5)
        
        print(f"DEBUG: Track Start: {start_time:.2f}h | End: {end_time:.2f}h")
        print(f"DEBUG: Calculated Split Time: {split_time:.2f}h")
        
        if split_time > end_time:
            print("ERROR: Split time is AFTER the track ends. (Are you at the end of the slider?)")
            return
        if split_time < start_time:
            print("ERROR: Split time is BEFORE the track starts. (Calculation error?)")
            return

        # 3. Determine New ID Name
        try:
            root_id = re.sub(r'[a-z]+$', '', self.current_id)
            print(f"DEBUG: Root ID detected: '{root_id}'")
            
            # Find existing in the whole DF
            existing_ids = self.df[self.df['ID'].str.startswith(root_id)]['ID'].unique()
            print(f"DEBUG: Existing Family IDs: {existing_ids}")
            
            suffixes = [uid.replace(root_id, '') for uid in existing_ids]
            letters = [s for s in suffixes if s.isalpha()]
            
            if not letters:
                next_char = 'b'
            else:
                max_char = max(letters)
                next_char = chr(ord(max_char) + 1)
                
            new_id = root_id + next_char
            print(f"DEBUG: Assigned New ID: {new_id}")
            
        except Exception as e:
            print(f"ERROR: Failed during ID generation: {e}")
            return

        # 4. Apply Split
        mask = (self.df['ID'] == self.current_id) & (self.df['Time (hours)'] >= split_time)
        rows_affected = mask.sum()
        print(f"DEBUG: Rows matched for move: {rows_affected}")
        
        if rows_affected == 0:
            print("WARNING: No rows matched the split criteria. Aborting.")
            return

        # EXECUTE SPLIT
        self.df.loc[mask, 'ID'] = new_id
        print(f"SUCCESS: Moved {rows_affected} rows to {new_id}")
        
        # 5. Recalculate Metrics
        try:
            self.refresh_metrics_for_ids([self.current_id, new_id])
        except Exception as e:
            print(f"ERROR: Metric recalculation failed: {e}")
            # We continue anyway to at least show the split
        
        # 6. Refresh
        print("DEBUG: Reloading sample...")
        self.load_sample()
        print("="*40 + "\n")


    def refresh_metrics_for_ids(self, id_list):
        """
        Runs the pipeline on specific IDs.
        """
        print(f"   -> RECALC: Running pipeline for {id_list}")
        
        mask = self.df['ID'].isin(id_list)
        if mask.sum() == 0:
            print("   -> ERROR: No rows found for these IDs to recalculate.")
            return

        sub_df = self.df[mask].copy()
        
        # Pipeline
        try:
            # 1. Smooth
            sub_df = process_mtb_area_relaxed(sub_df, window=10, spike_threshold=10.0)
            
            # 2. Fit Lowess
            sub_df = fit_lowess(sub_df, frac=0.25) 
            
            # 3. Metrics (with noise filter)
            sub_df = compute_doubling_metrics(sub_df, min_area=1.92, r2_threshold=0.7, min_doubling_time=4.0)
            
            # Update
            cols_to_update = [
                'Mtb Area Processed (µm)', 
                'Time Model (hours)', 
                'Mtb Area Model (µm)', 
                'r2', 
                'Doubling Amounts', 
                'Doubling Times'
            ]
            
            # Check if columns exist in sub_df (in case pipeline failed silently)
            available_cols = [c for c in cols_to_update if c in sub_df.columns]
            
            self.df.loc[mask, available_cols] = sub_df[available_cols]
            print("   -> RECALC: Complete.")
            
        except Exception as e:
            print(f"   -> RECALC ERROR: {e}")
            import traceback
            traceback.print_exc()

    # --- Update setup_bindings to include the new key ---
    def setup_bindings(self):
        self.viewer.bind_key('n', self.next_sample)
        self.viewer.bind_key('b', self.prev_sample)
        self.viewer.bind_key('s', self.save_df)
        self.viewer.bind_key('t', self.mark_transfer)
        self.viewer.bind_key('u', self.mark_uptake)
        self.viewer.bind_key('j', self.mark_junk)
        self.viewer.bind_key('g', self.mark_growth)
        self.viewer.bind_key('e', self.mark_edge)
        self.viewer.bind_key('k', self.split_track) # 'x' for cut/split

In [108]:
import napari
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import imageio.v3 as iio
import os
import io
import imageio  # This gives access to the v2 API
from skimage.transform import resize
import re  

# --- Configuration ---
VIDEO_DIR = '/Volumes/OPERA3/Nathan/data/macrohet/macrohet_results/glimpse_store/glimpses/'
SAVE_PATH = '/Volumes/OPERA3/Nathan/data/macrohet/macrohet_results/sc_df_alt_update.pkl'

# --- Ensure Style Defaults ---
sns.set_style("white")

class GlimpseAnnotator:
    def __init__(self, full_df, id_list):
        self.df = full_df         # Original DF with time-series lists
        self.id_list = id_list    # List of IDs to process
        self.current_idx = 0
        self.total = len(id_list)
        
        # Initialize Viewer
        self.viewer = napari.Viewer(title="Glimpse Annotator")
        
        # Setup bindings and load first sample
        self.setup_bindings()
        self.load_sample()

    def load_sample(self):
        """
        Loads video and plot. 
        AUTOMATICALLY SKIPS to the next ID if the video file is missing.
        """
        # Safety Check: Stop if we reached the end
        if self.current_idx >= self.total:
            print("All samples processed!")
            return

        self.current_id = self.id_list[self.current_idx]
        video_path = os.path.join(VIDEO_DIR, f'{self.current_id}.mp4')

        # --- SKIPPING LOGIC ---
        if not os.path.exists(video_path):
            print(f"⚠️ Missing Video for {self.current_id}. Skipping...")
            self.current_idx += 1
            # Recursively call this function again to try the next ID
            self.load_sample() 
            return
        # ----------------------
        
        # Clear existing layers
        self.viewer.layers.select_all()
        self.viewer.layers.remove_selected()
        
        print(f"--- Processing {self.current_idx + 1}/{self.total}: {self.current_id} ---")

        # 1. Attempt to Load Video
        video_loaded = False
        try:
            reader = imageio.get_reader(video_path, format='ffmpeg')
            video = np.stack([frame for frame in reader])
            reader.close()
            
            self.viewer.add_image(video, name=f'{self.current_id} Video')
            video_loaded = True
        except Exception as e:
            print(f"Warning: Could not read video for {self.current_id}. Error: {e}")

        # 2. Generate and Add Plot
        try:
            plot_img = self.create_plot_image(self.current_id)
            if plot_img is not None:
                self.viewer.add_image(plot_img, name='Growth Plot')
            else:
                print(f"No plot data generated for {self.current_id}")
        except Exception as e:
            print(f"Error generating plot for {self.current_id}: {e}")

        # 3. Layout and Time Reset
        self.viewer.grid.enabled = True
        self.viewer.grid.shape = (1, 2) if video_loaded else (1, 1)
        self.viewer.reset_view()
        
        # Reset Time Slider to 0 (t=0)
        if self.viewer.dims.ndim > 0:
            current_step = list(self.viewer.dims.current_step)
            current_step[0] = 0
            self.viewer.dims.current_step = tuple(current_step)

    def create_plot_image(self, ID):
        """
        Generates the Matplotlib graph using ROBUST Dynamic Baseline logic.
        Plots Doubling Grid (Light Grey) and Paired Labels (Dark Grey).
        """
        # 1. Filter and Sort
        sc_df = self.df[self.df['ID'] == ID].copy()
        if sc_df.empty: return None
        sc_df = sc_df.sort_values(by='Time Model (hours)')

        # Create Figure
        fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
        
        try:
            # 2. Extract Data & Metadata
            times = sc_df['Time Model (hours)'].values
            area_model = sc_df['Mtb Area Model (µm)'].values
            area_proc = sc_df['Mtb Area Processed (µm)'].values
            
            meta = sc_df.iloc[0]
            r2 = meta.get('r2', np.nan)
            strain = meta.get('Strain', 'Unknown')
            compound = meta.get('Compound', 'N/A')
            conc = meta.get('Concentration', 'N/A')
            
            # --- 3. Live Calculation Logic (Robust) ---
            
            # A. Helper Function
            def find_crossing(target, t_arr, a_arr):
                if not np.any(a_arr >= target): return None
                filled_a = np.nan_to_num(a_arr, nan=-np.inf)
                idx = np.argmax(filled_a >= target)
                if idx == 0 and filled_a[0] < target: return None
                if idx == 0: return t_arr[0]
                
                t1, t2 = t_arr[idx-1], t_arr[idx]
                a1, a2 = filled_a[idx-1], filled_a[idx]
                if a2 == a1: return t1
                return t1 + (t2 - t1) * ((target - a1) / (a2 - a1))

            # B. Determine Start Baseline
            raw_start_area = area_model[0] if not np.isnan(area_model[0]) else 1.92
            baseline = max(1.92, raw_start_area)
            
            # C. Determine Start Points
            if raw_start_area < baseline:
                t_start = find_crossing(baseline, times, area_model)
            else:
                t_start = times[0]
                
            if t_start is None:
                crossings_x = []
                crossings_y = []
            else:
                crossings_x = [t_start]
                crossings_y = [baseline]

            # D. Find Subsequent Crossings
            grid = [baseline * (2**i) for i in range(1, 6)]
            
            for target in grid:
                t_cross = find_crossing(target, times, area_model)
                if t_cross is not None:
                    if t_cross >= crossings_x[-1]:
                        crossings_x.append(t_cross)
                        crossings_y.append(target)
                    else:
                        break
                else:
                    break

            # --- 4. Plotting ---
            # Main Curves
            ax.plot(times, area_model, color='#d02c91', lw=2, label='Model')
            ax.scatter(times, area_proc, color='#1a9641', s=10, alpha=0.6, label='Data')
            
            # Draw Initial Baseline (Floor of first interval)
            if crossings_x:
                ax.axhline(y=baseline, color='lightgrey', linestyle='--', lw=1.5)
                ax.axvline(x=crossings_x[0], color='lightgrey', linestyle='--', lw=1.5)

            # Loop Intervals & Label
            for i in range(1, len(crossings_x)):
                t_prev, t_curr = crossings_x[i-1], crossings_x[i]
                y_prev, y_curr = crossings_y[i-1], crossings_y[i]
                
                # Draw Grid Lines (Ceiling)
                ax.axhline(y=y_curr, color='lightgrey', linestyle='--', lw=1.5)
                ax.axvline(x=t_curr, color='lightgrey', linestyle='--', lw=1.5)
                
                # Calculate Deltas
                dt = t_curr - t_prev
                delta_y = y_curr - y_prev
                
                # Label Logic: Left Justified, Sitting on the "Floor" line
                label_text = f"∆T={dt:.1f}h | ∆Mtb={delta_y:.2f}µm²"
                
                ax.text(t_prev + 0.2, 
                        y_prev * 1.02, # Sit on the previous line
                        label_text, 
                        color='#505050', # Dark Grey
                        fontsize=8, 
                        fontweight='bold', 
                        ha='left', 
                        va='bottom',
                        bbox=dict(boxstyle='square,pad=0.1', fc='white', ec='none', alpha=0.7))

            # --- 5. Formatting ---
            title_str = (f"ID: {ID}\n"
                         f"{strain} | {compound} {conc} | R2: {r2:.2f}")
            
            ax.set_title(title_str, fontsize=10)
            
            # Custom Legend
            from matplotlib.lines import Line2D
            custom_lines = [
                Line2D([0], [0], color='#d02c91', lw=2),
                Line2D([0], [0], color='#1a9641', marker='o', lw=0),
                Line2D([0], [0], color='lightgrey', linestyle='--', lw=1.5)
            ]
            ax.legend(custom_lines, ['Model', 'Data', 'Doubling Grid'], loc='upper left', fontsize=8)
            
            ax.set_xlabel("Time (hours)")
            ax.set_ylabel("Mtb Area (µm²)")
            sns.despine()
            
            # Convert to Image
            buf = io.BytesIO()
            fig.savefig(buf, format="png", bbox_inches='tight')
            plt.close(fig)
            buf.seek(0)
            img = imageio.v3.imread(buf, index=0)
            return img
            
        except Exception as e:
            print(f"Plotting error for {ID}: {e}")
            plt.close(fig)
            return None

    def next_sample(self, viewer):
        self.current_idx += 1
        self.load_sample()

    def prev_sample(self, viewer):
        if self.current_idx > 0:
            self.current_idx -= 1
            self.load_sample()

    # --- Classification Actions ---
    def update_origin(self, status):
        # Update the main DF
        self.df.loc[self.df['ID'] == self.current_id, 'mtb_origin'] = status
        print(f"Set {self.current_id} -> {status}")

    def mark_transfer(self, viewer): self.update_origin('Transfer')
    def mark_uptake(self, viewer): self.update_origin('Uptake')
    def mark_junk(self, viewer): self.update_origin('Junk')
    def mark_growth(self, viewer): self.update_origin('Growth')
    
    def mark_edge(self, viewer):
        self.df.loc[self.df['ID'] == self.current_id, 'Edge Status'] = True
        print(f"ID {self.current_id} marked as 'Edge Status'.")

    def save_df(self, viewer):
        print("Saving dataframe...")
        tmp_path = SAVE_PATH + '.tmp'
        self.df.to_pickle(tmp_path)
        os.replace(tmp_path, SAVE_PATH)
        print(f"Saved to {SAVE_PATH}")

    def split_track(self, viewer):
        """
        Splits the track with HEAVY debugging output to diagnose failures.
        """
        print("\n" + "="*40)
        print(f"DEBUG: Attempting Split on {self.current_id}")
        
        # 1. Get relative frame from slider
        try:
            current_frame = viewer.dims.current_step[0]
            print(f"DEBUG: Slider Frame: {current_frame}")
        except Exception as e:
            print(f"ERROR: Could not get frame from viewer: {e}")
            return
        
        # 2. Find Track Start
        track_df = self.df[self.df['ID'] == self.current_id]
        if track_df.empty:
            print(f"ERROR: ID {self.current_id} not found in DataFrame!")
            return
            
        start_time = track_df['Time (hours)'].min()
        end_time = track_df['Time (hours)'].max()
        split_time = start_time + (current_frame * 0.5)
        
        print(f"DEBUG: Track Start: {start_time:.2f}h | End: {end_time:.2f}h")
        print(f"DEBUG: Calculated Split Time: {split_time:.2f}h")
        
        if split_time > end_time:
            print("ERROR: Split time is AFTER the track ends. (Are you at the end of the slider?)")
            return
        if split_time < start_time:
            print("ERROR: Split time is BEFORE the track starts. (Calculation error?)")
            return

        # 3. Determine New ID Name
        try:
            root_id = re.sub(r'[a-z]+$', '', self.current_id)
            print(f"DEBUG: Root ID detected: '{root_id}'")
            
            # Find existing in the whole DF
            existing_ids = self.df[self.df['ID'].str.startswith(root_id)]['ID'].unique()
            print(f"DEBUG: Existing Family IDs: {existing_ids}")
            
            suffixes = [uid.replace(root_id, '') for uid in existing_ids]
            letters = [s for s in suffixes if s.isalpha()]
            
            if not letters:
                next_char = 'b'
            else:
                max_char = max(letters)
                next_char = chr(ord(max_char) + 1)
                
            new_id = root_id + next_char
            print(f"DEBUG: Assigned New ID: {new_id}")
            
        except Exception as e:
            print(f"ERROR: Failed during ID generation: {e}")
            return

        # 4. Apply Split
        mask = (self.df['ID'] == self.current_id) & (self.df['Time (hours)'] >= split_time)
        rows_affected = mask.sum()
        print(f"DEBUG: Rows matched for move: {rows_affected}")
        
        if rows_affected == 0:
            print("WARNING: No rows matched the split criteria. Aborting.")
            return

        # EXECUTE SPLIT
        self.df.loc[mask, 'ID'] = new_id
        print(f"SUCCESS: Moved {rows_affected} rows to {new_id}")
        
        # 5. Recalculate Metrics
        try:
            self.refresh_metrics_for_ids([self.current_id, new_id])
        except Exception as e:
            print(f"ERROR: Metric recalculation failed: {e}")
            # We continue anyway to at least show the split
        
        # 6. Refresh
        print("DEBUG: Reloading sample...")
        self.load_sample()
        print("="*40 + "\n")


    def refresh_metrics_for_ids(self, id_list):
        """
        Runs the pipeline on specific IDs.
        """
        print(f"   -> RECALC: Running pipeline for {id_list}")
        
        mask = self.df['ID'].isin(id_list)
        if mask.sum() == 0:
            print("   -> ERROR: No rows found for these IDs to recalculate.")
            return

        sub_df = self.df[mask].copy()
        
        # Pipeline
        try:
            # 1. Smooth
            sub_df = process_mtb_area_relaxed(sub_df, window=10, spike_threshold=10.0)
            
            # 2. Fit Lowess
            # Using robust version logic inline or calling the function if available
            sub_df = fit_lowess(sub_df, frac=0.25) 
            
            # 3. Metrics (with noise filter)
            sub_df = compute_doubling_metrics(sub_df, min_area=1.92, r2_threshold=0.7, min_doubling_time=4.0)
            
            # Update
            cols_to_update = [
                'Mtb Area Processed (µm)', 
                'Time Model (hours)', 
                'Mtb Area Model (µm)', 
                'r2', 
                'Doubling Amounts', 
                'Doubling Times'
            ]
            
            # Check if columns exist in sub_df (in case pipeline failed silently)
            available_cols = [c for c in cols_to_update if c in sub_df.columns]
            
            self.df.loc[mask, available_cols] = sub_df[available_cols]
            print("   -> RECALC: Complete.")
            
        except Exception as e:
            print(f"   -> RECALC ERROR: {e}")
            import traceback
            traceback.print_exc()

    # --- Update setup_bindings to include the new key ---
    def setup_bindings(self):
        self.viewer.bind_key('n', self.next_sample)
        self.viewer.bind_key('b', self.prev_sample)
        self.viewer.bind_key('s', self.save_df)
        self.viewer.bind_key('t', self.mark_transfer)
        self.viewer.bind_key('u', self.mark_uptake)
        self.viewer.bind_key('j', self.mark_junk)
        self.viewer.bind_key('g', self.mark_growth)
        self.viewer.bind_key('e', self.mark_edge)
        self.viewer.bind_key('k', self.split_track) # 'x' for cut/split

## Potential erroneous cells:

- 1969.3.6.ND4 - growth profile but unsure about doubling time 
- 5215.6.3.ND0004 - same
- 312.3.8.ND0004 - same
- 1872.3.6.ND0004 - same
- 1038.3.6.ND0004 - why is there a doubling time on this profile?
- 1309.3.7.ND0004 - same as above
- 256.3.3.ND0004 - same as above
- 968.3.4.ND0004 - where did the doubling time come from
- 2563.3.6.ND0004 - check segmentation
- 5382.3.7.ND0004 - why not more doubling times?
- 


##### Other notes: 
- 349.3.3.ND0004 - insane cording nearby

### Other to-do:

- check all under 6/7 hours? seems to be the threhsold for actual growth instances appearing
- generate all unfound videos and recheck those growth profiles 